In [ ]:
import yaml
from matplotlib import pyplot as plt

In [ ]:
version = '1.1.1'
all_logs = {
    'd01': {
        'Ours': [],
        # 'EmbedSeg': [],
        # 'Cellpose': []
    },
    'd02': {
        'Ours': [],
        # 'EmbedSeg': [],
        # 'Cellpose': [],
    }
}

In [ ]:
from matplotlib.colors import ListedColormap
import numpy as np

threshs = [0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9]
ap_keys = [f'ap_{t}' for t in threshs]

cmap = plt.get_cmap('Set2')
markers = ['o', '^', 's','*']
for dataset, logs in all_logs.items():

    fig, axs = plt.subplots(figsize=(4, 3))
    i = 0
    for model, model_logs_file in logs.items():
        if isinstance(model_logs_file,list):
            arr = []
            for j,mlf in enumerate(model_logs_file):
                arr.append([])
                xx = []
                with open(mlf, 'r') as f:
                    metrics = yaml.safe_load(f)

                for t, ap_key in zip(threshs, ap_keys):
                    try:
                        arr[j].append(metrics[ap_key])
                        xx.append(t)
                    except KeyError as e:
                        print(f"missing key {ap_key} for {model}, {model_logs_file}")
            arr = np.array(arr)
            avg = np.mean(arr,axis=0)
            std = np.std(arr,axis=0)
            axs.plot(xx, avg, label=model,
                 marker=markers[i], color=cmap(i), zorder=16-i)
            axs.fill_between(
                xx,avg-std,avg+std,alpha=0.2,
                color=cmap(i), zorder=8-i
            )

        else:
            print(f'not a list {model_logs_file}')
            with open(model_logs_file, 'r') as f:
                metrics = yaml.safe_load(f)

            arr = []
            xx = []
            for t, ap_key in zip(threshs, ap_keys):
                try:
                    arr.append(metrics[ap_key])
                    xx.append(t)
                except KeyError as e:
                    print(f"missing key {ap_key} for {model}, {model_logs_file}")

            axs.plot(xx, arr, label=model,
                 marker=markers[i], color=cmap(i), zorder=8-i)
        i += 1
    axs.set_xticks(threshs, threshs)
    axs.set_xlabel('IOU threshold')
    axs.set_ylabel(r'$\text{mAP}_3$')
    axs.spines[['right', 'top']].set_visible(False)
    axs.legend()
    fig.tight_layout()
    fig.savefig(f'../figs/metrics_{version}_{dataset}.pdf')

In [ ]:
#median image size
import numpy as np

report = {}

for dataset,logs in all_logs.items():

    model_logs_file = logs['Ours']
    if isinstance(model_logs_file,list):
        model_logs_file = model_logs_file[0]
    with open(model_logs_file,'r') as f:
        metrics = yaml.safe_load(f)

    median_h = np.median(metrics['image_h'])
    median_w = np.median(metrics['image_w'])

    print(f"[{dataset}] median size {median_h}, {median_w}")

    report[dataset] = dict(
        median_h =int(median_h),
        median_w=int(median_w)
    )

    for model,model_logs_file in logs.items():
        if isinstance(model_logs_file,list):
            model_logs_file = model_logs_file[0]
        with open(model_logs_file,'r') as f:
            metrics = yaml.safe_load(f)
        timing = metrics['mean_timing_total']
        if 'timing_total' in metrics:
            all_times = metrics['timing_total']
            std_time = np.std(all_times)
            print(f"[{dataset}] {model} avg time at inference {timing:.4f} (+-{std_time:.4f})")
        else:
            print(f"[{dataset}] {model} avg time at inference {timing:.4f}")
            std_time = None

        report[dataset][f'time_{model}_avg'] = float(timing)
        report[dataset][f'time_{model}_std'] = float(std_time) if std_time is not None else None

with open('../figs/timings.yaml','w',encoding='utf-8') as f:
    yaml.safe_dump(report,f)